In [ ]:
### Instalar ollama (32b — preferir A100)
!sudo apt update -q
!sudo apt install -y pciutils zstd -q
!curl -fsSL https://ollama.com/install.sh | sh

import threading, subprocess, time
thread = threading.Thread(target=lambda: subprocess.Popen(['ollama', 'serve']))
thread.start()
time.sleep(5)

!ollama pull qwen2.5:32b
!pip install ollama -q

!git clone https://github.com/average-peruvian/media-framing.git
!mv media-framing/* .
!rm -rf media-framing
!pip install -e . -q
!pip install hdbscan sentence-transformers -q

!wget -q https://github.com/average-peruvian/media-framing/releases/download/v0-alpha1/noticias_relevantes.csv
!wget -q https://github.com/average-peruvian/media-framing/releases/download/v0-alpha1/extracciones.csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/ant/selection/out'

from medianalysis.factual.grift    import EntityGrifter
from medianalysis.factual.embed    import build_embedding_input, MentionEmbedder
from medianalysis.factual.cluster  import cluster_generic
from medianalysis.factual.canonize import build_el_input, ELWorker

import json
import pandas as pd

In [ ]:
# Explosión
EntityGrifter(
    input_csv='extracciones.csv',
    output_csv=f'{DRIVE_OUT}/menciones_raw.csv',
    id_col='id', batch_size=100, resume=True,
).run()

In [ ]:
# Embeddings
build_embedding_input(
    menciones_csv=f'{DRIVE_OUT}/menciones_raw.csv',
    extracciones_csv='extracciones.csv',
    cuerpo_csv='noticias_relevantes.csv',
    output_csv=f'{DRIVE_OUT}/menciones_embedding_input.csv',
)

MentionEmbedder(
    input_csv=f'{DRIVE_OUT}/menciones_embedding_input.csv',
    output_csv=f'{DRIVE_OUT}/menciones_embeddings.csv',
    id_col='mention_id', batch_size=500, resume=True,
).run()

In [ ]:
# Clustering
type_df = pd.DataFrame([
    {'mention_id': f"{doc['id']}__{ent['id']}", 'type': ent['type']}
    for _, doc in pd.read_csv('extracciones.csv').iterrows()
    for ent in json.loads(doc['entities'])
])

cluster_generic(
    embeddings_csv=f'{DRIVE_OUT}/menciones_embeddings.csv',
    id_col='mention_id', output_csv=f'{DRIVE_OUT}/clusters.csv',
    prefix='ent', min_cluster_size=2,
    group_by_col='type', group_source_df=type_df,
)

In [ ]:
# Entity Linking
build_el_input(
    clusters_csv=f'{DRIVE_OUT}/clusters.csv',
    extracciones_csv='extracciones.csv',
    cuerpo_csv='noticias_relevantes.csv',
    output_csv=f'{DRIVE_OUT}/clusters_para_el.csv',
)

ELWorker(
    input_csv=f'{DRIVE_OUT}/clusters_para_el.csv',
    output_csv=f'{DRIVE_OUT}/entidades_canonicas.csv',
    id_col='cluster_id', batch_size=10, resume=True,
).run()